# LeetCode 1966 — Binary Searchable Numbers in an Unsorted Array bằng Python

Link đề: https://leetcode.com/problems/binary-searchable-numbers-in-an-unsorted-array/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search

> **Lưu ý (bài Premium 🔒):** đề trong notebook được viết lại theo hiểu biết về bài (nguồn tham khảo: bản mô tả của doocs/leetcode). Nếu bạn có tài khoản LeetCode Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc với đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def binarySearchableNumbers(self, nums: List[int]) -> int:
        ...
```

Notebook đi từ **mô phỏng mọi cách chọn pivot** (vét cạn đệ quy trên đoạn con) → nhận ra **tiêu chí đặc trưng** "mọi phần tử bên trái nhỏ hơn, mọi phần tử bên phải lớn hơn" → kiểm tra `O(n²)` → **max tiền tố + min hậu tố** `O(n)` → **ngăn xếp đơn điệu** một lượt.

## 1. Mục tiêu học

1. Hiểu thuật toán "tìm kiếm nhị phân với pivot ngẫu nhiên" trên dãy **chưa sắp** và khi nào nó **chắc chắn** tìm thấy một giá trị.
2. Chứng minh tiêu chí: `nums[i]` chắc chắn tìm thấy ⇔ `nums[i]` **lớn hơn mọi phần tử bên trái** và **nhỏ hơn mọi phần tử bên phải**.
3. Cài đặt tiêu chí bằng **max tiền tố** và **min hậu tố** trong `O(n)`.
4. Biết cách một lượt bằng **ngăn xếp đơn điệu** (giữ các ứng viên, loại khi gặp phần tử nhỏ hơn).
5. Kiểm thử bằng mô phỏng vét cạn trên mảng nhỏ; thảo luận follow-up khi có phần tử trùng.

## 2. Đề bài & ví dụ

Xét hàm giống tìm kiếm nhị phân, nhận `sequence` (dãy số nguyên) và `target`:

```
func(sequence, target)
  while sequence không rỗng
    chọn NGẪU NHIÊN một phần tử làm pivot
    nếu pivot = target, trả về true
    nếu pivot < target, xóa pivot và mọi phần tử BÊN TRÁI nó
    ngược lại, xóa pivot và mọi phần tử BÊN PHẢI nó
  trả về false
```

Khi dãy đã sắp xếp, hàm đúng với mọi giá trị. Khi chưa sắp, hàm có thể sai với một số giá trị. Cho mảng `nums` (các giá trị **khác nhau**), trả về **số giá trị được bảo đảm tìm thấy với mọi cách chọn pivot**.

**Ví dụ 1:** `nums = [7]` → `1`.

**Ví dụ 2:** `nums = [-1,5,2]` → `1`. Chỉ `-1` chắc chắn tìm thấy. Tìm `5`: nếu pivot là `2` (`2 < 5`) thì xóa `2` và mọi phần tử bên trái nó — `5` nằm bên trái `2` nên bị xóa theo ⇒ false. Tìm `2`: nếu pivot là `5` (`5 > 2`) thì xóa `5` và bên phải nó (gồm `2`) ⇒ false.

**Ràng buộc:** `1 ≤ nums.length ≤ 10⁵`, `−10⁵ ≤ nums[i] ≤ 10⁵`, các giá trị khác nhau.

**Follow-up:** nếu có phần tử trùng thì sửa thuật toán thế nào?

In [1]:
import random
import time
from functools import lru_cache
from typing import List

examples = [([7], 1), ([-1, 5, 2], 1)]

def run_once(seq, target, rnd):
    """Chạy hàm với pivot ngẫu nhiên (một lần)."""
    seq = list(seq)
    while seq:
        p = rnd.randrange(len(seq))
        if seq[p] == target:
            return True
        if seq[p] < target:
            seq = seq[p + 1:]
        else:
            seq = seq[:p]
    return False

rnd = random.Random(1)
for t in [-1, 5, 2]:
    results = {run_once([-1, 5, 2], t, rnd) for _ in range(200)}
    print(f"target={t}: các kết quả có thể quan sát được qua 200 lần chạy = {results}")

target=-1: các kết quả có thể quan sát được qua 200 lần chạy = {True}
target=5: các kết quả có thể quan sát được qua 200 lần chạy = {False, True}
target=2: các kết quả có thể quan sát được qua 200 lần chạy = {False, True}


## 3. Kiến thức nền

### 3.1 Mô phỏng "mọi lựa chọn" bằng đệ quy trên đoạn con

Dãy luôn còn lại là một **đoạn liên tiếp** `nums[l..r]` của mảng gốc (vì ta chỉ cắt bớt hai đầu). "Chắc chắn tìm thấy" ⇔ **mọi** lựa chọn pivot đều dẫn tới thành công ⇒ đệ quy với `all(...)`.

In [2]:
def guaranteed(nums, target):
    @lru_cache(maxsize=None)
    def ok(l, r):
        if l > r:
            return False
        for p in range(l, r + 1):
            if nums[p] == target:
                continue                          # chọn trúng: thành công
            if nums[p] < target:
                if not ok(p + 1, r): return False
            else:
                if not ok(l, p - 1): return False
        return True
    return ok(0, len(nums) - 1)

print([guaranteed((-1, 5, 2), t) for t in (-1, 5, 2)])

[True, False, False]


### 3.2 Max tiền tố và min hậu tố

`pre_max[i] = max(nums[0..i−1])` (−∞ nếu rỗng), `suf_min[i] = min(nums[i+1..])` (+∞ nếu rỗng). Dựng mỗi mảng trong một lượt.

In [3]:
nums = [1, 3, 2, 4, 6, 5, 7]
n = len(nums)
pre_max = [float('-inf')] * n
for i in range(1, n):
    pre_max[i] = max(pre_max[i - 1], nums[i - 1])
suf_min = [float('inf')] * n
for i in range(n - 2, -1, -1):
    suf_min[i] = min(suf_min[i + 1], nums[i + 1])
print("nums   :", nums)
print("pre_max:", pre_max)
print("suf_min:", suf_min)

nums   : [1, 3, 2, 4, 6, 5, 7]
pre_max: [-inf, 1, 3, 3, 4, 6, 6]
suf_min: [2, 2, 4, 5, 5, 7, inf]


### 3.3 Ngăn xếp đơn điệu

Giữ các phần tử trong ngăn xếp theo thứ tự **tăng dần**; khi gặp phần tử nhỏ hơn đỉnh, lấy đỉnh ra. Mỗi phần tử vào/ra một lần ⇒ `O(n)`.

In [4]:
st = []
for x in [3, 1, 4, 1, 5, 9, 2, 6]:
    while st and st[-1] > x:
        st.pop()
    st.append(x)
    print(f"x={x}: ngăn xếp = {st}")

x=3: ngăn xếp = [3]
x=1: ngăn xếp = [1]
x=4: ngăn xếp = [1, 4]
x=1: ngăn xếp = [1, 1]
x=5: ngăn xếp = [1, 1, 5]
x=9: ngăn xếp = [1, 1, 5, 9]
x=2: ngăn xếp = [1, 1, 2]
x=6: ngăn xếp = [1, 1, 2, 6]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2 `[-1, 5, 2]`:
- `5` có phần tử **bên phải nhỏ hơn** nó (`2`). Nếu `2` được chọn làm pivot (`2 < 5`), thuật toán xóa `2` **và mọi thứ bên trái** — mất `5`.
- `2` có phần tử **bên trái lớn hơn** nó (`5`). Nếu `5` được chọn (`5 > 2`), xóa `5` **và mọi thứ bên phải** — mất `2`.
- `-1`: mọi phần tử bên phải đều lớn hơn ⇒ pivot nào khác `-1` cũng lớn hơn ⇒ xóa phía **phải** pivot, không bao giờ đụng tới `-1` ⇒ chắc chắn tìm thấy.

Tiêu chí: `nums[i]` **chắc chắn** tìm thấy ⇔ `max(nums[0..i−1]) < nums[i] < min(nums[i+1..])`.

Tính tay `[1, 3, 2, 4, 6, 5, 7]`: thỏa mãn là `1` (bên phải đều lớn hơn), `4` (trái `{1,3,2}` < 4 < phải `{6,5,7}`), `7`. `3` hỏng vì bên phải có `2`; `2` hỏng vì bên trái có `3`; tương tự `6`, `5`. ⇒ đáp án `3`.

In [5]:
nums = [1, 3, 2, 4, 6, 5, 7]
for i, x in enumerate(nums):
    left, right = nums[:i], nums[i + 1:]
    cond = all(y < x for y in left) and all(y > x for y in right)
    print(f"{x}: trái {left}, phải {right} -> {'✔' if cond else '✘'} | mô phỏng: {guaranteed(tuple(nums), x)}")

1: trái [], phải [3, 2, 4, 6, 5, 7] -> ✔ | mô phỏng: True
3: trái [1], phải [2, 4, 6, 5, 7] -> ✘ | mô phỏng: False
2: trái [1, 3], phải [4, 6, 5, 7] -> ✘ | mô phỏng: False
4: trái [1, 3, 2], phải [6, 5, 7] -> ✔ | mô phỏng: True
6: trái [1, 3, 2, 4], phải [5, 7] -> ✘ | mô phỏng: False
5: trái [1, 3, 2, 4, 6], phải [7] -> ✘ | mô phỏng: False
7: trái [1, 3, 2, 4, 6, 5], phải [] -> ✔ | mô phỏng: True


## 5. Cách 1 — Brute Force / baseline: mô phỏng mọi cách chọn pivot

Với mỗi giá trị, chạy đệ quy `guaranteed` trên mọi đoạn `(l, r)` có ghi nhớ.

In [6]:
def count_brute(nums):
    t = tuple(nums)
    return sum(guaranteed(t, x) for x in nums)

for nums_, exp in examples:
    print(count_brute(nums_), "(kỳ vọng", exp, ")")
print("[1,3,2,4,6,5,7] ->", count_brute([1, 3, 2, 4, 6, 5, 7]))

1 (kỳ vọng 1 )
1 (kỳ vọng 1 )
[1,3,2,4,6,5,7] -> 3


**Độ phức tạp:** mỗi giá trị có `O(n²)` đoạn, mỗi đoạn thử `O(n)` pivot ⇒ `O(n³)` mỗi giá trị, `O(n⁴)` tổng. Chỉ cho `n ≤ 12`.

**Nút thắt:** cần một **đặc trưng đơn giản** thay cho mô phỏng.

## 6. Ý tưởng thuật toán / bước đột phá

**Định lý:** `x = nums[i]` chắc chắn tìm thấy ⇔ mọi `j < i` có `nums[j] < x` **và** mọi `j > i` có `nums[j] > x`.

- **Nếu** có `j > i` với `nums[j] < x` **thì** chọn ngay pivot `nums[j]` ở bước đầu: `pivot < x` ⇒ xóa pivot và mọi phần tử bên trái — gồm `x` ⇒ thất bại. Tương tự với `j < i`, `nums[j] > x`.
- **Nếu** điều kiện thỏa **thì** mỗi pivot khác `x`: nếu nó ở bên trái `x` thì nó nhỏ hơn `x` ⇒ xóa phần bên trái pivot (không chứa `x`); nếu ở bên phải thì lớn hơn ⇒ xóa phần bên phải (không chứa `x`). `x` không bao giờ bị xóa, dãy giảm dần ⇒ cuối cùng `x` được chọn.

**Cài đặt:** đếm các `i` với `pre_max[i] < nums[i] < suf_min[i]`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Kiểm tra tiêu chí trực tiếp `O(n²)`

In [7]:
def count_criterion_quadratic(nums):
    n = len(nums)
    cnt = 0
    for i, x in enumerate(nums):
        if all(nums[j] < x for j in range(i)) and all(nums[j] > x for j in range(i + 1, n)):
            cnt += 1
    return cnt

for nums_, exp in examples:
    print(count_criterion_quadratic(nums_), "(kỳ vọng", exp, ")")

1 (kỳ vọng 1 )
1 (kỳ vọng 1 )


**Độ phức tạp:** `O(n²)`.

### 7.2 Cách 3 — Max tiền tố + min hậu tố (lời giải chính)

In [8]:
def count_prefix_suffix(nums):
    n = len(nums)
    ok = [True] * n
    mx = float('-inf')
    for i, x in enumerate(nums):           # trái -> phải: có phần tử lớn hơn ở bên trái?
        if x < mx:
            ok[i] = False
        else:
            mx = x
    mn = float('inf')
    for i in range(n - 1, -1, -1):          # phải -> trái: có phần tử nhỏ hơn ở bên phải?
        if nums[i] > mn:
            ok[i] = False
        else:
            mn = nums[i]
    return sum(ok)

for nums_, exp in examples:
    print(count_prefix_suffix(nums_), "(kỳ vọng", exp, ")")

1 (kỳ vọng 1 )
1 (kỳ vọng 1 )


Vì giá trị khác nhau, `x < mx` ⇔ có phần tử bên trái lớn hơn; khi `x ≥ mx` thì `x` là max mới. **Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

### 7.3 Cách 4 — Ngăn xếp đơn điệu một lượt

Duyệt trái → phải, giữ `mx` (max đã thấy) và ngăn xếp `cand` các **ứng viên** (những phần tử từng là max tiền tố, và chưa bị phần tử nhỏ hơn phía sau "đánh bại").
- Gặp `x`: lấy khỏi `cand` mọi ứng viên `> x` (vì chúng có phần tử bên phải nhỏ hơn).
- **Nếu** `x > mx` **thì** `x` là ứng viên mới (đẩy vào), cập nhật `mx`.
Kết thúc: `len(cand)` là đáp án. Ngăn xếp luôn **tăng dần** (các max tiền tố tăng dần).

In [9]:
def count_stack(nums):
    cand = []
    mx = float('-inf')
    for x in nums:
        while cand and cand[-1] > x:
            cand.pop()
        if x > mx:
            cand.append(x)
            mx = x
    return len(cand)

for nums_, exp in examples:
    print(count_stack(nums_), "(kỳ vọng", exp, ")")

1 (kỳ vọng 1 )
1 (kỳ vọng 1 )


**Độ phức tạp:** `O(n)` thời gian (mỗi phần tử vào/ra tối đa một lần), `O(n)` bộ nhớ xấu nhất (mảng tăng dần).

## 8. TRACE từng bước

Trace cách 3 trên `nums = [1, 3, 2, 4, 6, 5, 7]`:

| i | x | mx trước | lượt trái → phải | mn trước | lượt phải → trái | ok |
|---|---|---|---|---|---|---|
| 0 | 1 | −∞ | ✔ (mx=1) | 2 | ✔ | ✔ |
| 1 | 3 | 1 | ✔ (mx=3) | 2 | ✘ (3 > 2) | ✘ |
| 2 | 2 | 3 | ✘ | 4 | ✔ | ✘ |
| 3 | 4 | 3 | ✔ | 5 | ✔ | ✔ |
| 4 | 6 | 4 | ✔ | 5 | ✘ | ✘ |
| 5 | 5 | 6 | ✘ | 7 | ✔ | ✘ |
| 6 | 7 | 6 | ✔ | +∞ | ✔ | ✔ |

In [10]:
def trace_prefix_suffix(nums):
    n = len(nums); ok = [True] * n
    mx = float('-inf')
    print("Lượt trái -> phải:")
    for i, x in enumerate(nums):
        if x < mx:
            ok[i] = False; print(f"  i={i} x={x}: {x} < mx={mx} -> loại")
        else:
            print(f"  i={i} x={x}: là max tiền tố mới"); mx = x
    mn = float('inf')
    print("Lượt phải -> trái:")
    for i in range(n - 1, -1, -1):
        if nums[i] > mn:
            ok[i] = False; print(f"  i={i} x={nums[i]}: {nums[i]} > mn={mn} -> loại")
        else:
            print(f"  i={i} x={nums[i]}: là min hậu tố mới"); mn = nums[i]
    print("ok =", ok, "-> đáp án", sum(ok))
    return sum(ok)

trace_prefix_suffix([1, 3, 2, 4, 6, 5, 7])

Lượt trái -> phải:
  i=0 x=1: là max tiền tố mới
  i=1 x=3: là max tiền tố mới
  i=2 x=2: 2 < mx=3 -> loại
  i=3 x=4: là max tiền tố mới
  i=4 x=6: là max tiền tố mới
  i=5 x=5: 5 < mx=6 -> loại
  i=6 x=7: là max tiền tố mới
Lượt phải -> trái:
  i=6 x=7: là min hậu tố mới
  i=5 x=5: là min hậu tố mới
  i=4 x=6: 6 > mn=5 -> loại
  i=3 x=4: là min hậu tố mới
  i=2 x=2: là min hậu tố mới
  i=1 x=3: 3 > mn=2 -> loại
  i=0 x=1: là min hậu tố mới
ok = [True, False, False, True, False, False, True] -> đáp án 3


3

In [11]:
def trace_stack(nums):
    cand, mx = [], float('-inf')
    for x in nums:
        popped = []
        while cand and cand[-1] > x:
            popped.append(cand.pop())
        pushed = x > mx
        if pushed:
            cand.append(x); mx = x
        print(f"  x={x:>2}: loại {popped if popped else '-'}, {'đẩy vào' if pushed else 'không đẩy'} -> ứng viên {cand}")
    return len(cand)

print("đáp án:", trace_stack([1, 3, 2, 4, 6, 5, 7]))

  x= 1: loại -, đẩy vào -> ứng viên [1]
  x= 3: loại -, đẩy vào -> ứng viên [1, 3]
  x= 2: loại [3], không đẩy -> ứng viên [1]
  x= 4: loại -, đẩy vào -> ứng viên [1, 4]
  x= 6: loại -, đẩy vào -> ứng viên [1, 4, 6]
  x= 5: loại [6], không đẩy -> ứng viên [1, 4]
  x= 7: loại -, đẩy vào -> ứng viên [1, 4, 7]
đáp án: 3


## 9. Code nộp LeetCode

In [12]:
class Solution:
    def binarySearchableNumbers(self, nums: List[int]) -> int:
        n = len(nums)
        ok = [True] * n
        mx = float('-inf')
        for i, x in enumerate(nums):
            if x < mx:
                ok[i] = False                 # có phần tử lớn hơn ở bên trái
            else:
                mx = x
        mn = float('inf')
        for i in range(n - 1, -1, -1):
            if nums[i] > mn:
                ok[i] = False                 # có phần tử nhỏ hơn ở bên phải
            else:
                mn = nums[i]
        return sum(ok)

for nums_, exp in examples:
    assert Solution().binarySearchableNumbers(nums_) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Định lý (mục 6)** — chứng minh đầy đủ hai chiều:

(⇒) Giả sử `x = nums[i]` chắc chắn tìm thấy. Nếu tồn tại `j > i` với `nums[j] < x`: ở bước đầu (dãy là cả mảng) có thể chọn pivot `nums[j]`; vì `nums[j] < x` nên xóa `nums[0..j]` ⊇ `{x}` ⇒ `x` mất ⇒ trả `false` — mâu thuẫn. Tương tự với `j < i`, `nums[j] > x`.

(⇐) Giả sử mọi phần tử bên trái `< x`, bên phải `> x`. Bất biến: dãy hiện tại là một đoạn chứa `x`. Một pivot `p ≠ x` nằm bên trái `x` thì `p < x` ⇒ xóa `p` và các phần tử bên trái nó — vẫn giữ `x`; nằm bên phải thì `p > x` ⇒ xóa `p` và bên phải nó — vẫn giữ `x`. Mỗi bước dãy ngắn đi ít nhất 1, không bao giờ rỗng khi còn `x` ⇒ cuối cùng `x` được chọn ⇒ `true`.

**Cài đặt.** Lượt trái → phải loại đúng các `i` có phần tử bên trái lớn hơn (`x < max tiền tố`); lượt phải → trái loại các `i` có phần tử bên phải nhỏ hơn. Phần còn lại đúng là các phần tử thỏa định lý.

**Ngăn xếp.** Ứng viên phải là max tiền tố tại thời điểm xuất hiện (`x > mx`); nó bị loại đúng khi gặp phần tử nhỏ hơn ở phía sau. Ngăn xếp tăng dần nên chỉ cần so với đỉnh.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng mọi pivot | `O(n⁴)` | `O(n²)` |
| 2. Tiêu chí trực tiếp | `O(n²)` | `O(1)` |
| 3. Max tiền tố + min hậu tố | `O(n)` | `O(n)` |
| 4. Ngăn xếp đơn điệu | `O(n)` | `O(n)` |

In [13]:
rnd = random.Random(1966)
big = rnd.sample(range(-10**5, 10**5 + 1), 10**5)
almost = sorted(big)
for i in range(0, 10**5 - 1, 50):
    almost[i], almost[i + 1] = almost[i + 1], almost[i]          # gần như đã sắp
for name, fn in [("prefix_suffix", count_prefix_suffix), ("stack", count_stack), ("Solution", Solution().binarySearchableNumbers)]:
    t0 = time.perf_counter(); a = fn(big); b = fn(almost); dt = time.perf_counter() - t0
    print(f"{name:<14}: ngẫu nhiên -> {a}, gần sắp -> {b}  {dt * 1000:6.1f} ms")

prefix_suffix : ngẫu nhiên -> 0, gần sắp -> 96000    12.5 ms
stack         : ngẫu nhiên -> 0, gần sắp -> 96000     7.2 ms
Solution      : ngẫu nhiên -> 0, gần sắp -> 96000    13.5 ms


## 12. Bộ test

In [14]:
ALL = {
    "quadratic": count_criterion_quadratic,
    "prefix_suffix": count_prefix_suffix,
    "stack": count_stack,
    "Solution": lambda a: Solution().binarySearchableNumbers(a),
}

def check(nums, exp=None, brute=True):
    ref = count_brute(nums) if brute else count_criterion_quadratic(nums)
    for name, fn in ALL.items():
        assert fn(list(nums)) == ref, (name, nums)
    if exp is not None:
        assert ref == exp, (nums, ref, exp)
    return ref

for nums_, exp in examples:
    check(nums_, exp)
print("Ví dụ đề: OK")

check([1, 2, 3, 4, 5], 5)        # đã sắp: mọi giá trị
check([5, 4, 3, 2, 1], 0)        # giảm dần: không giá trị nào (n > 1)
check([2, 1], 0)
check([1, 3, 2, 4, 6, 5, 7], 3)  # tự dựng (mục 4)
check([3, 1, 2, 5, 4, 6], 1)     # tự dựng: chỉ 6
print("Edge case: OK")

rnd = random.Random(2006)
for _ in range(1500):
    n = rnd.randint(1, 8)
    check(rnd.sample(range(-20, 21), n))
print("1500 mảng nhỏ so với mô phỏng mọi pivot: OK")

for _ in range(30):
    n = rnd.randint(1, 1500)
    check(rnd.sample(range(-10**5, 10**5 + 1), n), brute=False)
print("30 mảng cỡ vừa so với tiêu chí O(n^2): OK")

Ví dụ đề: OK
Edge case: OK
1500 mảng nhỏ so với mô phỏng mọi pivot: OK


30 mảng cỡ vừa so với tiêu chí O(n^2): OK


Các edge case "tự dựng" ở trên được xác nhận bằng mô phỏng mọi pivot trong `check` (đối chiếu lại đề gốc nếu muốn).

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ kiểm một phía (chỉ max tiền tố)

In [15]:
def count_bug_one_side(nums):
    mx, cnt = float('-inf'), 0
    for x in nums:
        if x > mx:
            cnt += 1; mx = x                 # SAI: quên kiểm phía phải
    return cnt

nums_ = [-1, 5, 2]
print("sai :", count_bug_one_side(nums_), "| đúng:", Solution().binarySearchableNumbers(nums_))

sai : 2 | đúng: 1


**Vì sao sai:** `5` là max tiền tố nhưng bên phải có `2 < 5` ⇒ có thể bị xóa. **Sửa:** kiểm cả min hậu tố (hoặc dùng ngăn xếp để loại).

### Lỗi 2 — Đếm số phần tử đúng vị trí của nó trong mảng đã sắp

In [16]:
def count_bug_sorted_position(nums):
    s = sorted(nums)
    return sum(a == b for a, b in zip(nums, s))     # SAI: "đúng chỗ" không đủ

nums_ = [3, 2, 1]
print("sai :", count_bug_sorted_position(nums_), "| đúng:", Solution().binarySearchableNumbers(nums_))

sai : 1 | đúng: 0


**Vì sao sai:** `2` đứng đúng vị trí giữa của mảng đã sắp, nhưng bên trái có `3 > 2` (và bên phải có `1 < 2`). "Đúng vị trí" là điều kiện **cần** (khi giá trị khác nhau, phần tử thỏa định lý phải đúng vị trí) nhưng **không đủ**. **Sửa:** dùng đúng tiêu chí max/min.

### Lỗi 3 — Ngăn xếp: đẩy mọi phần tử (không yêu cầu là max tiền tố)

In [17]:
def count_bug_stack_push_all(nums):
    cand = []
    for x in nums:
        while cand and cand[-1] > x:
            cand.pop()
        cand.append(x)                        # SAI: x có thể có phần tử lớn hơn ở bên trái
    return len(cand)

nums_ = [3, 1, 2]
print("sai :", count_bug_stack_push_all(nums_), "| đúng:", Solution().binarySearchableNumbers(nums_))

sai : 2 | đúng: 0


**Vì sao sai:** sau khi `1` loại `3`, ngăn xếp có `[1]` rồi `[1, 2]` — nhưng `1`, `2` đều có `3` lớn hơn ở bên trái. **Sửa:** chỉ đẩy khi `x > mx` (max của **toàn bộ** tiền tố, không chỉ của ngăn xếp).

### Lỗi 4 — Khởi tạo `mx = 0`, `mn = 0` (quên giá trị âm)

In [18]:
def count_bug_init_zero(nums):
    n = len(nums); ok = [True] * n
    mx = 0                                   # SAI: nums có thể âm
    for i, x in enumerate(nums):
        if x < mx: ok[i] = False
        else: mx = x
    mn = 0                                   # SAI: nums có thể dương
    for i in range(n - 1, -1, -1):
        if nums[i] > mn: ok[i] = False
        else: mn = nums[i]
    return sum(ok)

for a in ([-5, -3, -1], [1, 2, 3], [7]):
    print(f"{a}: sai = {count_bug_init_zero(a)}, đúng = {Solution().binarySearchableNumbers(a)}")

[-5, -3, -1]: sai = 0, đúng = 3
[1, 2, 3]: sai = 0, đúng = 3
[7]: sai = 0, đúng = 1


**Vì sao sai:** `−10⁵ ≤ nums[i] ≤ 10⁵`; khởi tạo bằng 0 coi như có một phần tử "ảo" bằng 0 ở hai đầu. **Sửa:** `mx = −∞`, `mn = +∞` (hoặc giá trị ngoài miền như `−10⁶`, `10⁶`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Phần tử lớn hơn mọi phần tử bên trái và nhỏ hơn mọi phần tử bên phải" ⇒ **max tiền tố + min hậu tố** (hai lượt) hoặc **ngăn xếp đơn điệu** (một lượt).
- Bài mô tả một **thuật toán ngẫu nhiên** và hỏi "chắc chắn" ⇒ nghĩ theo **đối thủ xấu nhất** chọn pivot; tìm điều kiện cần bằng cách chọn pivot "phá hoại" ngay bước đầu.
- Mảng con còn lại luôn là **đoạn liên tiếp** ⇒ brute force bằng đệ quy `(l, r)` có ghi nhớ.
- Liên quan: **LeetCode 769/768 — Max Chunks To Make Sorted** (dùng max tiền tố/min hậu tố tương tự).

## 15. Biến thể của bài

1. **Có phần tử trùng (follow-up):** chọn trúng **bất kỳ bản sao** cũng thắng, và một giá trị có thể xuất hiện ở nhiều vị trí — tiêu chí "một vị trí thỏa trái `≤ x`, phải `≥ x`" **không còn đủ** (ô dưới tìm được phản ví dụ). Cách an toàn: lập luận theo **giá trị** — giá trị `v` thất bại nếu đối thủ chọn được pivot loại hết mọi bản sao; nên kiểm bằng mô phỏng như `guaranteed_value`.
2. **Liệt kê các giá trị** (thay vì đếm).
3. **LeetCode 915 — Partition Array into Disjoint Intervals:** max tiền tố ≤ min hậu tố tại điểm cắt.
4. **LeetCode 769 — Max Chunks To Make Sorted:** đếm điểm cắt `i` với `max(nums[..i]) ≤ min(nums[i+1..])`.
5. **Pivot luôn là phần tử giữa** (tìm kiếm nhị phân thật trên mảng chưa sắp): đường đi xác định, kiểm từng giá trị trong `O(log n)`.

In [19]:
def guaranteed_value(nums, v):
    """Có trùng: giá trị v chắc chắn tìm thấy (chọn trúng bản sao nào cũng thắng)."""
    return guaranteed(tuple(nums), v)

def naive_dup(nums):
    """Áp nguyên công thức của bài (đếm GIÁ TRỊ khác nhau thỏa điều kiện tại ít nhất một vị trí)."""
    n = len(nums); good = set()
    for i, x in enumerate(nums):
        if all(y <= x for y in nums[:i]) and all(y >= x for y in nums[i + 1:]):
            good.add(x)
    return len(good)

rnd = random.Random(15)
found = None
for _ in range(5000):
    a = [rnd.randint(1, 4) for _ in range(rnd.randint(1, 7))]
    truth = sum(guaranteed_value(a, v) for v in set(a))
    if truth != naive_dup(a):
        found = (a, truth, naive_dup(a)); break
print("Follow-up có trùng — phản ví dụ cho công thức ngây thơ:", found)

def max_chunks_to_sorted(arr):
    """LeetCode 768 (có trùng): số điểm cắt i (sau i) với max tiền tố <= min hậu tố, cộng 1."""
    n = len(arr); suf = [float('inf')] * (n + 1)
    for i in range(n - 1, -1, -1):
        suf[i] = min(suf[i + 1], arr[i])
    mx, chunks = float('-inf'), 0
    for i in range(n):
        mx = max(mx, arr[i])
        if mx <= suf[i + 1]:
            chunks += 1
    return chunks

print("768 [2,1,3,4,4] ->", max_chunks_to_sorted([2, 1, 3, 4, 4]), "| [5,4,3,2,1] ->", max_chunks_to_sorted([5, 4, 3, 2, 1]))

Follow-up có trùng — phản ví dụ cho công thức ngây thơ: ([3, 4, 1, 3], 1, 0)
768 [2,1,3,4,4] -> 4 | [5,4,3,2,1] -> 1


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | đệ quy mọi pivot trên `(l, r)` | `O(n⁴)` | `O(n²)` | kiểm chứng |
| 2. Tiêu chí trực tiếp | so với mọi phần tử hai bên | `O(n²)` | `O(1)` | hiểu bài |
| 3. Max/min hai lượt | `pre_max < x < suf_min` | `O(n)` | `O(n)` | **nộp bài** |
| 4. Ngăn xếp đơn điệu | ứng viên tăng dần, loại khi gặp nhỏ hơn | `O(n)` | `O(n)` | một lượt, stream |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().binarySearchableNumbers([2, 1, 3, 5, 4, 6])` bằng bao nhiêu? Những giá trị nào?

**Bài 2 (tính tay):** Với `nums = [4, 1, 2, 3, 5]`, chỉ ra cho mỗi giá trị **không** chắc chắn một pivot "phá hoại" ở bước đầu.

**Bài 3 (điền TODO):** Hoàn thiện `searchable_values(nums)` trả về **danh sách giá trị** chắc chắn tìm thấy (theo thứ tự xuất hiện).

**Bài 4 (tự viết lại):** Viết lại cách 3 chỉ dùng **một** mảng `suf_min` (min hậu tố) và một biến `mx`, không cần mảng `ok`.

**Bài 5 (mini challenge):** Biến thể "pivot luôn là phần tử **giữa**" (`p = (l + r) // 2`) thay vì ngẫu nhiên. Viết `count_middle_pivot(nums)` đếm số giá trị được tìm thấy, `O(n log n)`.

In [20]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def searchable_values(nums):
    # TODO: như Solution nhưng trả [nums[i] for i if ok[i]]
    pass

# Bài 4 — TODO
def count_one_array(nums):
    # TODO: suf_min dựng trước; duyệt trái->phải với mx, đếm khi mx < x < suf_min[i+1]
    pass

# Bài 5 — TODO
def count_middle_pivot(nums):
    # TODO: với mỗi x mô phỏng tìm kiếm nhị phân xác định trên (l, r)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Max tiền tố: `2, 3, 5, 6`.
- Mức 2: Loại các phần tử có phần tử nhỏ hơn bên phải: `2` (có 1), `5` (có 4).
- Mức 3: Còn `3` và `6` ⇒ `2`.

**Bài 2**
- Mức 1: `4` có `1` nhỏ hơn bên phải.
- Mức 2: `1, 2, 3` có `4` lớn hơn bên trái.
- Mức 3: Pivot phá hoại: với `4` chọn `1`; với `1, 2, 3` chọn `4`.

**Bài 3**
- Mức 1: Hai lượt như `Solution`.
- Mức 2: Gom theo thứ tự chỉ số.
- Mức 3: `len(...)` phải bằng `Solution()`.

**Bài 4**
- Mức 1: `suf_min[i]` = min của `nums[i..]`.
- Mức 2: Điều kiện với `i`: `mx < nums[i] < suf_min[i + 1]`.
- Mức 3: Cập nhật `mx` sau khi xét `i`.

**Bài 5**
- Mức 1: Đường đi xác định, mỗi giá trị `O(log n)` bước.
- Mức 2: Dùng `l, r` như tìm kiếm nhị phân cổ điển.
- Mức 3: So với mô phỏng trực tiếp trên list.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1 & 3
def searchable_values_sol(nums):
    n = len(nums); ok = [True] * n
    mx = float('-inf')
    for i, x in enumerate(nums):
        if x < mx: ok[i] = False
        else: mx = x
    mn = float('inf')
    for i in range(n - 1, -1, -1):
        if nums[i] > mn: ok[i] = False
        else: mn = nums[i]
    return [nums[i] for i in range(n) if ok[i]]

r1 = searchable_values_sol([2, 1, 3, 5, 4, 6])
print("Bài 1/3:", len(r1), r1)
assert r1 == [3, 6] and Solution().binarySearchableNumbers([2, 1, 3, 5, 4, 6]) == 2

# Bài 2
nums2 = [4, 1, 2, 3, 5]
for i, x in enumerate(nums2):
    bad = [y for y in nums2[i + 1:] if y < x] + [y for y in nums2[:i] if y > x]
    print(f"  {x}: {'chắc chắn' if not bad else 'pivot phá hoại: ' + str(bad[0])}")
assert searchable_values_sol(nums2) == [5]

Bài 1/3: 2 [3, 6]
  4: pivot phá hoại: 1
  1: pivot phá hoại: 4
  2: pivot phá hoại: 4
  3: pivot phá hoại: 4
  5: chắc chắn


In [22]:
# Bài 4
def count_one_array_sol(nums):
    n = len(nums)
    suf = [float('inf')] * (n + 1)
    for i in range(n - 1, -1, -1):
        suf[i] = min(suf[i + 1], nums[i])
    mx, cnt = float('-inf'), 0
    for i, x in enumerate(nums):
        if mx < x < suf[i + 1]:
            cnt += 1
        mx = max(mx, x)
    return cnt

# Bài 5
def count_middle_pivot_sol(nums):
    n = len(nums); cnt = 0
    for x in nums:
        l, r = 0, n - 1
        while l <= r:
            m = (l + r) // 2
            if nums[m] == x:
                cnt += 1; break
            if nums[m] < x: l = m + 1
            else: r = m - 1
    return cnt

def middle_brute(nums):
    cnt = 0
    for x in nums:
        seq = list(nums)
        while seq:
            p = (len(seq) - 1) // 2
            if seq[p] == x: cnt += 1; break
            seq = seq[p + 1:] if seq[p] < x else seq[:p]
    return cnt

rnd = random.Random(5)
for _ in range(1500):
    a = rnd.sample(range(-30, 31), rnd.randint(1, 12))
    assert count_one_array_sol(a) == Solution().binarySearchableNumbers(a)
    assert count_middle_pivot_sol(a) == middle_brute(a)
    assert count_middle_pivot_sol(a) >= Solution().binarySearchableNumbers(a)   # chắc chắn với mọi pivot => cũng thắng với pivot giữa
print("Bài 4, Bài 5 OK | pivot giữa với [-1,5,2]:", count_middle_pivot_sol([-1, 5, 2]))

Bài 4, Bài 5 OK | pivot giữa với [-1,5,2]: 2


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "lớn hơn mọi bên trái, nhỏ hơn mọi bên phải":**

```python
ok = [True] * n
mx = -inf
for i, x in enumerate(nums):           # trái -> phải
    if x < mx: ok[i] = False
    else: mx = x
mn = inf
for i in range(n - 1, -1, -1):         # phải -> trái
    if nums[i] > mn: ok[i] = False
    else: mn = nums[i]
return sum(ok)
```

**Checklist:**
- [ ] Kiểm **cả hai** phía (max trái, min phải)?
- [ ] Ngăn xếp: chỉ đẩy phần tử là **max tiền tố**?
- [ ] Mảng một phần tử ⇒ `1`?
- [ ] Đã so với mô phỏng mọi pivot trên mảng nhỏ?
- [ ] (Follow-up) phần tử trùng: kiểm lại bằng brute force?
- [ ] Chữ ký đúng: `def binarySearchableNumbers(self, nums: List[int]) -> int`?